# Création de la base d'apprentissage

## 1. Présentation générale

Ce notebook constitue l'étape cruciale de notre projet : la **réconciliation de données multi-sources** pour bâtir notre dataset d'entraînement. L'objectif est d'aligner les statistiques de performance des joueurs avec leurs valeurs marchandes respectives.

Nous centralisons alors dans un premier temps des fichiers issus de l'API Soccerdata, regroupant ainsi des données FBref et des données Understat. De plus, nous centralisons également des données Transfermarkt (les données financières ainsi que notre variable cible : la valeur marchande) et du mapping issu de worldfootballR.

Nous réalisons ensuite une fusion à plusieurs niveaux : nous utilisons les identifiants connus des joueurs puisdu fuzzy-mapping.


Nous devrions obtenir finalement une table prête pour de plus profondes analyses voire pour de la modélisation, mêlant ainsi des données issues des performances sportives à des données analysant la valeur marchande des joueurs de football.

Pour ce faire, nous importons dans un premier temps des packages et des fonctions nécessaires à la création de notre base d'apprentissage.

In [1]:
# Importation des packages nécessaires

import pandas as pd
import os
import sys

# On connecte le notebook à tous les fichiers inclus dans le dossier /fonctions
sys.path.append(os.path.abspath("../fonctions"))

%load_ext autoreload
%autoreload 2

from merging import *

## 2. Chargement et préparation des sources


Nous importons dans un premier temps nos trois fichiers comprenant nos données :
- issues du mapping de worldfootballR
- issues de transfermarkt
- issues de soccerdata

In [2]:
# Chargement des données du mapping
df_mapping_initial = pd.read_csv("../data_finale/mapping_worldfootballR/mapping_fbref_tm.csv", encoding='latin1')

# Chargement des données du dataset Soccerdata
df_soccerdata_initial = pd.read_csv("../data/soccerdata/data_final_soccerdata.csv")

# Chargement des données du dataset Transfermarkt
df_players = pd.read_csv("../data/transfermarkt_datasets/players.csv")
df_valuations = pd.read_csv("../data/transfermarkt_datasets/player_valuations.csv")

# Préparation des données de Transfermarkt
df_tm_initial = prepare_transfermarkt_data(
    df_players,
    df_valuations
)

# Chargement des données du dataset de blessures Transfermarkt
df_blessures = pd.read_csv("../data/dataset_blessures.csv")

# Préparation des données de blessures Transfermarkt
df_blessures_initial = aggregate_injuries_by_season(df_blessures)

Plutôt que de traiter chaque dataframe manuellement ici, nous utilisons la fonction match_player_data. Cette fonction encapsule toute la logique de nettoyage définie précédemment :

- Correction de l'encoding : Application de fix_encoding sur les noms FBref.

- Normalisation des noms : Suppression des accents, mise en minuscule et nettoyage des caractères spéciaux via normalize_name.

- Harmonisation des dates : Extraction de l'année de naissance (dob_key) pour faciliter le matching entre les sources.

- Création de clés composites : Génération de clés basées sur "Prénom + Nom" pour Transfermarkt.

In [3]:
# Nous appliquons les logiques décrites ci-dessus
df_mapping, df_soccerdata, df_tm, df_blessures = match_player_data(df_mapping_initial, df_soccerdata_initial,
                                                      df_tm_initial, df_blessures_initial)

## 3. La fusion à multi-niveaux

Nous appliquons ensuite une stratégie de fusion des bases de données en 2 étapes pour maximiser le taux de correspondance.

Dans un premier temps, nous réalisons une jointure exacte via le dictionnaire de mapping.

Ensuite, nous effectuons une recherche plus floue (fuzzy) sur le mapping avec un seuil supérieur à 90%.

In [4]:
df_final, still_missing = run_player_matching(df_soccerdata, df_mapping, df_tm, df_blessures)

[1] Match exact Mapping saison : 16280 | Restants : 833
[2] Match fuzzy Mapping saison : 139 | Restants : 694
[3.1] Match direct TM exact : 296 | Restants : 398
[3.2] Match fuzzy TM : 92 | Restants : 305

Classement rempli : 14216 / 16807
NA restants (2591) — clubs hors top 5 ligues cette saison (relégués, D2, etc.) : ['Alavés', 'Angers', 'Arsenal', 'Aston Villa', 'Atalanta', 'Athletic Club', 'Atlético Madrid', 'Augsburg', 'Auxerre', 'Barcelona', 'Bayern Munich', 'Bologna', 'Bournemouth', 'Brentford', 'Brest', 'Brighton', 'Burnley', 'Cagliari', 'Celta Vigo', 'Chelsea', 'Como', 'Cremonese', 'Crystal Palace', 'Dortmund', 'Eintracht Frankfurt', 'Elche', 'Espanyol', 'Everton', 'Fiorentina', 'Freiburg', 'Fulham', 'Genoa', 'Getafe', 'Girona', 'Gladbach', 'Hamburger SV', 'Heidenheim', 'Hellas Verona', 'Hoffenheim', 'Inter', 'Juventus', 'Köln', 'Lazio', 'Le Havre', 'Lecce', 'Leeds United', 'Lens', 'Levante', 'Leverkusen', 'Lille', 'Liverpool', 'Lorient', 'Lyon', 'Mainz 05', 'Mallorca', 'Manche

In [5]:
# On filtre les lignes valides pour comparer les années sans risquer le plantage d'antécédents
df_valide = df_final[df_final['valuation_season_year'].notna()]
erreurs_saisons = df_valide[df_valide['season_year'].astype(int) != df_valide['valuation_season_year'].astype(int)]


print(f"Nombre d'incohérences de saison (conflits d'années) : {len(erreurs_saisons)}")
if not erreurs_saisons.empty:
    print(erreurs_saisons[['player', 'season_year', 'valuation_season_year', 'team', 'match_method']].head().to_string(index=False))
else:
    print("Aucun conflit d'année détecté sur les données jointes.")

Nombre d'incohérences de saison (conflits d'années) : 0
Aucun conflit d'année détecté sur les données jointes.


Nous pouvons enfin importer notre base d'apprentissage sous le format CSV.

In [6]:
df_final.to_csv(r'..\data_finale\base_apprentissage.csv', index=False, sep=',', encoding='utf-8-sig')
still_missing.to_csv(r'..\data_finale\analyse_orphelins\still_missing.csv', index=False, sep=',', encoding='utf-8-sig')
df_soccerdata.to_csv(r'..\data_finale\analyse_orphelins\soccerdata.csv', index=False, sep=',', encoding='utf-8-sig')

In [7]:
df_final

,league,season,team,player,nation,pos,age,born,Playing Time_MP,Playing Time_Starts,...,injury_minor_unknown_nb_m,injury_musculaire,injury_genou,injury_cheville_pied,injury_mollet_tibia,injury_dos_bassin,injury_trauma_severe,injury_medical_repos,injury_minor_unknown,classement
0,ENG-Premier League,2021,Arsenal,Ainsley Maitland-Niles,ENG,"MF,DF",22,1997.0,11,5,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,8.0
1,ENG-Premier League,2021,Arsenal,Alexandre Lacazette,FRA,FW,29,1991.0,31,22,...,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,8.0
2,ENG-Premier League,2021,Arsenal,Bernd Leno,GER,GK,28,1992.0,35,35,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0
3,ENG-Premier League,2021,Arsenal,Bukayo Saka,ENG,MF,18,2001.0,32,30,...,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,8.0
4,ENG-Premier League,2021,Arsenal,Calum Chambers,ENG,DF,25,1995.0,10,8,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16802,ITA-Serie A,2526,Juventus,Vasilije Adžić,MNE,MF,19-357,2006.0,10,1,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,NaN
16803,ITA-Serie A,2526,Parma,Ben Cremaschi,USA,MF,21-063,2005.0,8,2,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
16804,ITA-Serie A,2526,Pisa,İsak Vural,TUR,MF,19-341,2006.0,12,7,...,3.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,NaN
16805,ITA-Serie A,2526,Roma,Jan Ziółkowski,POL,DF,20-333,2005.0,15,5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN


## **Les joueurs orphelins**

In [11]:
still_missing = pd.read_csv(r'..\data_finale\analyse_orphelins\still_missing.csv', encoding='utf-8-sig')
df_soccerdata = pd.read_csv(r'..\data_finale\analyse_orphelins\soccerdata.csv', encoding='utf-8-sig')

In [12]:
# Prépare les mêmes structures que dans la fonction pour comprendre pourquoi ça n'a pas matché
remaining_debug = still_missing.copy()
remaining_debug['join_key_clean'] = remaining_debug['join_key'].apply(clean_text)
remaining_debug['team_clean'] = remaining_debug['team'].apply(clean_text)

# Prépare le côté TM
cols_to_extract = ['name', 'player_id']
if 'current_club_name' in df_tm.columns:
    cols_to_extract.append('current_club_name')

df_tm_unique = df_tm[cols_to_extract].drop_duplicates(subset=['name']).copy()
df_tm_unique['name_clean'] = df_tm_unique['name'].apply(clean_text)
if 'current_club_name' in df_tm_unique.columns:
    df_tm_unique['club_clean'] = df_tm_unique['current_club_name'].apply(clean_text)
df_tm_unique = df_tm_unique.drop_duplicates(subset=['name_clean'])

tm_dict = df_tm_unique.set_index('name_clean').to_dict(orient='index')
tm_keys = [k for k in tm_dict.keys() if k and isinstance(k, str)]

rows_diagnostic = []

for _, row in remaining_debug.iterrows():
    current_key = row['join_key_clean']
    current_team = row['team_clean']
    if not current_key:
        continue

    try:
        res_set  = process.extractOne(current_key, tm_keys, scorer=fuzz.token_set_ratio)
        res_sort = process.extractOne(current_key, tm_keys, scorer=fuzz.token_sort_ratio)
    except Exception:
        continue

    if res_set and res_sort:
        res = res_set if res_set[1] >= res_sort[1] else res_sort
    elif res_set:
        res = res_set
    elif res_sort:
        res = res_sort
    else:
        rows_diagnostic.append({
            'player_sd': row['player'],
            'team_sd': row['team'],
            'season_year': row['season_year'],
            'best_match_tm': 'Aucun candidat',
            'club_tm': '',
            'score_nom': 0,
            'score_club': 0,
            'raison_rejet': 'Aucun résultat fuzzy'
        })
        continue

    meta_tm = tm_dict.get(res[0], {})
    score_club = fuzz.token_set_ratio(current_team, meta_tm.get('club_clean', ''))
    seuil_club = 70 if res[1] < 90 else 50

    if res[1] < 90 and score_club < seuil_club and res[1] < 95:
        raison = f"Score nom OK ({res[1]}) mais club trop faible ({score_club} < {seuil_club})"
    elif res[1] < 50:
        raison = f"Score nom trop faible ({res[1]})"
    else:
        raison = f"Score nom ({res[1]}) + club ({score_club}) — à vérifier manuellement"

    rows_diagnostic.append({
        'player_sd': row['player'],
        'team_sd': row['team'],
        'season_year': row['season_year'],
        'best_match_tm': meta_tm.get('name', res[0]),
        'club_tm': meta_tm.get('current_club_name', ''),
        'score_nom': res[1],
        'score_club': score_club,
        'raison_rejet': raison
    })

df_diagnostic = pd.DataFrame(rows_diagnostic).sort_values('score_nom', ascending=False).reset_index(drop=True)
display(df_diagnostic)

,player_sd,team_sd,season_year,best_match_tm,club_tm,score_nom,score_club,raison_rejet
0,Yellu Santiago,Valencia,2021,Yellu Santiago,Futebol Clube de Arouca,100.000000,19.354839,Score nom (100.0) + club (19.354838709677423) ...
1,Juanma Herzog,Las Palmas,2024,Juanma Herzog,UD Las Palmas,100.000000,100.000000,Score nom (100.0) + club (100.0) — à vérifier ...
2,Joselu Pérez,Getafe,2025,Joselu,Real Madrid Club de Fútbol,100.000000,25.000000,Score nom (100.0) + club (25.0) — à vérifier m...
3,Fabio Moreno Fell,Mainz 05,2025,Fábio,Football Club de Nantes,100.000000,19.354839,Score nom (100.0) + club (19.354838709677423) ...
4,Pablo Saenz,Granada,2023,Pablo,Clube de Regatas do Flamengo,100.000000,22.857143,Score nom (100.0) + club (22.85714285714286) —...
...,...,...,...,...,...,...,...,...
300,Dženan Pejčinović,Wolfsburg,2022,Nathan de Medina,Fudbalski Klub Partizan Beograd,41.666667,25.000000,Score nom OK (41.666666666666664) mais club tr...
301,Dženan Pejčinović,Wolfsburg,2025,Nathan de Medina,Fudbalski Klub Partizan Beograd,41.666667,25.000000,Score nom OK (41.666666666666664) mais club tr...
302,Bertuğ Yıldırım,Rennes,2025,Dimitry Bertaud,NaN,40.000000,0.000000,Score nom OK (40.0) mais club trop faible (0.0...
303,Bertuğ Yıldırım,Rennes,2023,Dimitry Bertaud,NaN,40.000000,0.000000,Score nom OK (40.0) mais club trop faible (0.0...


In [13]:
df_diagnostic.to_csv("../notebooks/orphelins.csv")

In [14]:
nb_joueurs_orphelins = still_missing['join_key'].nunique()
nb_joueurs_total = df_soccerdata['join_key'].nunique()

print(f"Joueurs orphelins : {nb_joueurs_orphelins}")
print(f"Taux joueurs orphelins : {nb_joueurs_orphelins / nb_joueurs_total:.2%}")

Joueurs orphelins : 284
Taux joueurs orphelins : 4.59%


In [15]:
orphelins_par_saison = (
    still_missing
    .groupby('season_year')
    .size()
    .sort_index()
)

print(orphelins_par_saison)

season_year
2020      5
2021     19
2022     32
2023     50
2024     16
2025    183
dtype: int64


In [16]:
temp_data = still_missing[still_missing['season_year']<2025]
temp_data

,league,season,team,player,nation,pos,age,born,Playing Time_MP,Playing Time_Starts,...,Performance_PKcon,Performance_OG,xg,xa,np_xg,xg_chain,xg_buildup,join_key,dob_key,season_year
0,ENG-Premier League,2021,Manchester Utd,Will Fish,ENG,DF,17,2003.0,1,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,will fish,2003,2020
1,ENG-Premier League,2223,Brighton,Cameron Peupion,AUS,MF,19,2002.0,1,0,...,NaN,0,0.053613,0.0,0.053613,0.000000,0.000000,cameron peupion,2002,2022
2,ENG-Premier League,2223,Liverpool,Ben Gannon-Doak,SCO,MF,16,2005.0,2,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,ben gannon doak,2005,2022
3,ENG-Premier League,2223,Manchester City,Shea Charles,NIR,"DF,MF",18,2003.0,1,0,...,NaN,0,0.000000,0.0,0.000000,0.129427,0.129427,shea charles,2003,2022
4,ENG-Premier League,2223,Nottingham Forest,Alex Mighten,ENG,"FW,MF",20,2002.0,1,0,...,NaN,0,0.000000,0.0,0.000000,0.000000,0.000000,alex mighten,2002,2022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
279,ITA-Serie A,2324,Salernitana,Mateusz Łęgowski,POL,MF,20,2003.0,29,10,...,NaN,0,NaN,NaN,NaN,NaN,NaN,mateusz u0141 u0119gowski,2003,2023
280,ITA-Serie A,2324,Salernitana,Triantafyllos Pasaridīs,GRE,DF,27,1996.0,8,5,...,NaN,0,NaN,NaN,NaN,NaN,NaN,triantafyllos pasarid u012bs,1996,2023
281,ITA-Serie A,2324,Udinese,Antonio Tikvić,CRO,DF,19,2004.0,1,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,antonio tikvi u0107,2004,2023
282,ITA-Serie A,2425,Inter,Luka Topalović,SVN,MF,18,2006.0,1,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,luka topalovi u0107,2006,2024
